# ==========================================================
# PURE NORMAL-ONLY AUTOENCODER
# Training data:
#   - ONLY files from the true normal folder/state
#
# Evaluation/anomaly data:
#   - motor fully off
#   - motor being switched on
#   - motor being switched off
#   - electrical fault
#   - mechanical fault
#   - thermal fault
#
# Cross-validation is SESSION/FILE-WISE.
# No off, transition, or fault file is used to fit the scaler or autoencoder.
# ==========================================================

In [ ]:
# ==========================================================
# CELL 0 — install dependencies
# ==========================================================
%pip install pandas numpy scikit-learn joblib openpyxl matplotlib tensorflow

In [ ]:
# ==========================================================
# CELL 1 — imports
# ==========================================================
import os
import glob
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    StratifiedGroupKFold,
    StratifiedShuffleSplit,
)


In [ ]:
# ==========================================================
# CELL 2 — configuration
# ==========================================================
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

BASE_PATH = (
    r"../../data/Motor_data_model"
)

SCALE_DIVISOR = 1_000_000.0

FEATURE_COLUMNS = [
    "T_slope_10s",
    "PWM",
    "Irms",
    "Vrms",
    "Current_Variance",
    "Current_Crest_Factor",
    "Vib_RMS",
    "Vib_Min",
    "Vib_Q1",
    "Vib_Median",
    "Vib_Q3",
    "Vib_Max",
    "Vib_Variance",
    "Vib_Skewness",
]

# Kept ONLY to label/organize sessions (motor_off, transitions,
# normal, faults). There is no classifier head that predicts
# these — they are used purely for grouping, stratified CV
# splits, and choosing which reconstruction threshold applies.
FINAL_LABELS = [
    "motor_off",
    "turning_off",
    "turning_on",
    "normal",
    "electrical_fault",
    "mechanical_fault",
    "thermal_fault",
]

LABEL_TO_ID = {
    label: index
    for index, label in enumerate(FINAL_LABELS)
}

SAFE_LABELS = [
    "motor_off",
    "turning_off",
    "turning_on",
    "normal",
]

FAULT_LABELS = [
    "electrical_fault",
    "mechanical_fault",
    "thermal_fault",
]

FAULT_CLASS_IDS = np.asarray(
    [LABEL_TO_ID[label] for label in FAULT_LABELS],
    dtype=np.int64,
)

N_FEATURES = len(FEATURE_COLUMNS)
N_CLASSES = len(FINAL_LABELS)

WINDOW_LEN = 30
WINDOW_STEP = 10
ROW_PERIOD_SEC = 0.015

N_SPLITS = 5
BATCH_SIZE = 128
MAX_EPOCHS = 180
LEARNING_RATE = 1e-3

# Compact pure-MLP autoencoder architecture (same sizing
# family as the hybrid model's reconstruction branch).
ENCODER_DENSE_1 = 32
ENCODER_DENSE_2 = 16
DROPOUT_LAYER_1 = 0.10
DROPOUT_LAYER_2 = 0.05
LATENT_DENSE_UNITS = 4
DECODER_DENSE_1 = 16
DECODER_DENSE_2 = 32

RECON_QUANTILE = 0.990

PWM_BIN_EDGES = np.asarray([
    0.00, 0.20, 0.35, 0.45, 0.55, 0.65, 0.75, 1.01,
], dtype=np.float32)

PWM_BIN_NAMES = [
    "0_20", "20_35", "35_45", "45_55", "55_65", "65_75", "75_100",
]

LOAD_PROXY_FEATURE = "Irms"
LOAD_SPLIT_QUANTILE = 0.50
MIN_THRESHOLD_WINDOWS = 30

LOAD_STATE_NAMES = ["low_load", "high_load"]

OUTPUT_DIR = "pure_ae_hybrid_structure_output"
CV_DIR = os.path.join(OUTPUT_DIR, "cross_validation")
FINAL_DIR = os.path.join(OUTPUT_DIR, "final_model")
MANUAL_TEST_DIR = os.path.join(OUTPUT_DIR, "manual_test_results")

for folder in [CV_DIR, FINAL_DIR, MANUAL_TEST_DIR]:
    os.makedirs(folder, exist_ok=True)

print("Model: Pure Dense Autoencoder (reconstruction-only)")
print("Input:", WINDOW_LEN, "x", N_FEATURES)
print("Session labels (grouping only):", FINAL_LABELS)
print("Trained ONLY on normal-with-load windows.")

In [ ]:
# ==========================================================
# CELL 3 — deterministic file-to-class mapping
# (identical to the hybrid model — used only for session
#  grouping / reconstruction-reference selection, never fed
#  into the autoencoder as a target.)
# ==========================================================
def normalized_path(path):
    return str(path).replace("\\", "/").lower()


def label_for(path):
    path_text = normalized_path(path)
    stem = Path(path).stem.lower()

    if any(term in path_text for term in [
        "motor_being_off", "turning_off", "transition_off", "off_spikes",
    ]):
        return "turning_off"

    if any(term in path_text for term in [
        "motor_being_on", "turning_on", "transition_on", "on_spikes",
    ]):
        return "turning_on"

    if "/motor_off/" in path_text or "motor_off" in stem:
        return "motor_off"

    if any(term in path_text for term in [
        "electrical fault", "electrical_fault", "abnormal_current",
    ]):
        return "electrical_fault"

    if any(term in path_text for term in [
        "mechanical fault", "mechanical_fault", "abnormal_vibration",
    ]):
        return "mechanical_fault"

    if any(term in path_text for term in [
        "thermal fault", "thermal_fault", "abnormal_temp",
    ]):
        return "thermal_fault"

    if (
        "/normal/" in path_text
        or stem.startswith("normal_")
        or stem == "normal"
    ):
        return "normal"

    return None


def anomaly_target_from_class_id(class_id):
    # Used only for reporting/evaluation, never trained on.
    return int(class_id in FAULT_CLASS_IDS)


def is_normal_with_load_file(path, label):
    if label != "normal":
        return False

    path_text = normalized_path(path)

    without_load_terms = [
        "without_load", "without load", "no_load", "no load",
        "unloaded", "normal_noload", "normal_no_load",
    ]

    with_load_terms = [
        "with_load", "with load", "loaded", "normal_load",
    ]

    if any(term in path_text for term in without_load_terms):
        return False

    if any(term in path_text for term in with_load_terms):
        return True

    raise ValueError(
        "Cannot identify loaded/unloaded normal file: "
        f"{path}"
    )

In [ ]:
# ==========================================================
# CELL 4 — discover and read files
# ==========================================================
def discover_files(base_path):
    files = glob.glob(os.path.join(base_path, "**", "*.xlsx"), recursive=True)
    files += glob.glob(os.path.join(base_path, "**", "*.csv"), recursive=True)
    return sorted(set(files))


def read_one(path):
    path = Path(path)

    if path.suffix.lower() == ".xlsx":
        df = pd.read_excel(path)
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported file type: {path}")

    missing_columns = [
        column for column in FEATURE_COLUMNS if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing columns in {path.name}: {missing_columns}"
        )

    clean = df[FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce")
    clean = clean / SCALE_DIVISOR
    clean = clean.replace([np.inf, -np.inf], np.nan)
    clean = clean.dropna().reset_index(drop=True)

    if len(clean) < WINDOW_LEN:
        raise ValueError(
            f"{path.name} has only {len(clean)} valid rows."
        )

    return clean

In [ ]:
# ==========================================================
# CELL 5 — load labeled sessions
# ==========================================================
files = discover_files(BASE_PATH)

if not files:
    raise RuntimeError(f"No CSV or Excel files found under: {BASE_PATH}")

sessions = []

for session_id, file_path in enumerate(files):
    label = label_for(file_path)

    if label is None:
        print("SKIP unrecognized:", os.path.relpath(file_path, BASE_PATH))
        continue

    try:
        df = read_one(file_path)
    except Exception as error:
        print(
            "SKIP read error:",
            os.path.relpath(file_path, BASE_PATH),
            "|",
            error,
        )
        continue

    class_id = LABEL_TO_ID[label]

    sessions.append({
        "session_id": session_id,
        "X": df.to_numpy(dtype=np.float32),
        "file": os.path.relpath(file_path, BASE_PATH),
        "label": label,
        "class_id": class_id,
        "anomaly": anomaly_target_from_class_id(class_id),
        "reconstruction_reference": int(
            is_normal_with_load_file(file_path, label)
        ),
        "rows": len(df),
    })

if not sessions:
    raise RuntimeError("No labeled sessions were loaded.")

class_counts = pd.Series([
    session["label"] for session in sessions
]).value_counts().reindex(FINAL_LABELS, fill_value=0)

print("\nFILES PER CLASS")
print(class_counts)

missing_classes = [
    label for label in FINAL_LABELS if class_counts[label] == 0
]

if missing_classes:
    raise RuntimeError(f"No files found for classes: {missing_classes}")

loaded_normal_files = [
    session["file"] for session in sessions
    if session["reconstruction_reference"] == 1
]

if not loaded_normal_files:
    raise RuntimeError("No normal-with-load files were identified.")

print("\nNORMAL WITH LOAD — the ONLY files used to train the autoencoder")
for file_name in loaded_normal_files:
    print("  ", file_name)


In [ ]:
# ==========================================================
# CELL 6 — create 30 x 14 windows
# ==========================================================
def make_windows_from_array(X_raw):
    starts = np.arange(
        0, len(X_raw) - WINDOW_LEN + 1, WINDOW_STEP, dtype=np.int64,
    )

    if len(starts) == 0:
        return None, None

    windows = np.stack([
        X_raw[start:start + WINDOW_LEN] for start in starts
    ]).astype(np.float32)

    return windows, starts


def build_dataset_from_sessions(selected_sessions):
    all_windows = []
    all_class_labels = []
    all_anomaly_labels = []
    all_files = []
    all_starts = []
    all_groups = []
    all_reconstruction_reference = []

    for session in selected_sessions:
        windows, starts = make_windows_from_array(session["X"])

        if windows is None:
            continue

        number_of_windows = len(windows)

        all_windows.append(windows)
        all_class_labels.extend([session["class_id"]] * number_of_windows)
        all_anomaly_labels.extend([session["anomaly"]] * number_of_windows)
        all_files.extend([session["file"]] * number_of_windows)
        all_starts.extend(starts.tolist())
        all_groups.extend([session["session_id"]] * number_of_windows)
        all_reconstruction_reference.extend(
            [session["reconstruction_reference"]] * number_of_windows
        )

    if not all_windows:
        raise RuntimeError("No windows were created.")

    return (
        np.concatenate(all_windows, axis=0).astype(np.float32),
        np.asarray(all_class_labels, dtype=np.int64),
        np.asarray(all_anomaly_labels, dtype=np.float32),
        np.asarray(all_reconstruction_reference, dtype=np.float32),
        np.asarray(all_files, dtype=object),
        np.asarray(all_starts, dtype=np.int64),
        np.asarray(all_groups, dtype=np.int64),
    )


In [ ]:
# ==========================================================
# CELL 7 — normalization utilities
# ==========================================================
def calculate_normalization_statistics(X_train):
    flattened_rows = X_train.reshape(-1, N_FEATURES)

    mean = np.mean(flattened_rows, axis=0).astype(np.float32)
    variance = np.var(flattened_rows, axis=0).astype(np.float32)
    variance = np.maximum(variance, 1e-12)

    return mean, variance


def normalize_windows_numpy(X, mean, variance):
    return (
        (X - mean.reshape(1, 1, N_FEATURES))
        / np.sqrt(variance.reshape(1, 1, N_FEATURES) + 1e-7)
    ).astype(np.float32)

In [ ]:
# ==========================================================
# CELL 8 — pure Dense autoencoder model (NO anomaly head,
# NO classification head — reconstruction only)
# ==========================================================
def window_mse_loss(y_true, y_pred):
    squared_error = tf.square(y_true - y_pred)
    return tf.reduce_mean(squared_error, axis=[1, 2])


def build_autoencoder_model(normalization_mean, normalization_variance):
    input_layer = tf.keras.Input(
        shape=(WINDOW_LEN, N_FEATURES),
        name="motor_window",
    )

    normalizer = tf.keras.layers.Normalization(
        axis=-1,
        mean=normalization_mean,
        variance=normalization_variance,
        name="input_normalization",
    )
    normalized = normalizer(input_layer)

    flattened = tf.keras.layers.Flatten(name="flatten_window")(normalized)

    encoded = tf.keras.layers.Dense(
        ENCODER_DENSE_1, activation="relu", name="encoder_dense1",
    )(flattened)
    encoded = tf.keras.layers.Dropout(DROPOUT_LAYER_1, name="enc_dropout_1")(encoded)
    encoded = tf.keras.layers.Dense(
        ENCODER_DENSE_2, activation="relu", name="encoder_dense2",
    )(encoded)
    encoded = tf.keras.layers.Dropout(DROPOUT_LAYER_2, name="enc_dropout_2")(encoded)
    latent = tf.keras.layers.Dense(
        LATENT_DENSE_UNITS, activation="relu", name="latent_dense",
    )(encoded)

    decoded = tf.keras.layers.Dense(
        DECODER_DENSE_1, activation="relu", name="decoder_dense1",
    )(latent)
    decoded = tf.keras.layers.Dense(
        DECODER_DENSE_2, activation="relu", name="decoder_dense2",
    )(decoded)

    reconstruction_flat = tf.keras.layers.Dense(
        WINDOW_LEN * N_FEATURES, activation="linear", name="reconstruction_dense",
    )(decoded)

    reconstruction_output = tf.keras.layers.Reshape(
        (WINDOW_LEN, N_FEATURES), name="reconstruction_output",
    )(reconstruction_flat)

    model = tf.keras.Model(
        inputs=input_layer,
        outputs=reconstruction_output,
        name="pure_dense_autoencoder",
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=window_mse_loss,
    )

    return model


def print_model_size(model):
    total_parameters = model.count_params()
    float32_weight_kb = total_parameters * 4 / 1024
    estimated_int8_weight_kb = total_parameters / 1024

    print("\nMODEL SIZE")
    print("Total parameters:", total_parameters)
    print("Estimated Float32 weight size KB:", round(float32_weight_kb, 2))
    print("Estimated INT8 weight size KB:", round(estimated_int8_weight_kb, 2))


_model_preview = build_autoencoder_model(
    normalization_mean=np.zeros(N_FEATURES, dtype=np.float32),
    normalization_variance=np.ones(N_FEATURES, dtype=np.float32),
)
_model_preview.summary()
print_model_size(_model_preview)
del _model_preview
tf.keras.backend.clear_session()


In [ ]:
# ==========================================================
# CELL 9 — training sample weights and dataset split helpers
# (only the reconstruction-reference weighting remains —
#  no class or anomaly sample weights.)
# ==========================================================
def make_training_sample_weights(reconstruction_reference):
    # Only loaded-normal windows train the reconstruction AE.
    return reconstruction_reference.astype(np.float32)


def split_train_validation_by_group(
    X, y_class, reconstruction_reference, files, starts, groups,
):
    """
    Create validation data at SESSION level, never by overlapping
    windows, to avoid leakage. y_class is used ONLY to stratify the
    group split so every state is represented in validation — it is
    never fed to the model.
    """
    unique_groups, first_indices = np.unique(groups, return_index=True)
    group_labels = y_class[first_indices]

    class_group_counts = pd.Series(group_labels).value_counts()

    if class_group_counts.min() < 2:
        raise RuntimeError(
            "Each class needs at least 2 sessions inside the training "
            "fold to create a stratified session-wise validation split. "
            f"Current session counts: {class_group_counts.to_dict()}"
        )

    validation_fraction = max(0.15, N_CLASSES / len(unique_groups))
    validation_fraction = min(validation_fraction, 0.40)

    splitter = StratifiedShuffleSplit(
        n_splits=1, test_size=validation_fraction, random_state=SEED,
    )

    group_train_pos, group_validation_pos = next(
        splitter.split(unique_groups, group_labels)
    )

    train_group_ids = unique_groups[group_train_pos]
    validation_group_ids = unique_groups[group_validation_pos]

    train_index = np.flatnonzero(np.isin(groups, train_group_ids))
    validation_index = np.flatnonzero(np.isin(groups, validation_group_ids))

    if np.intersect1d(groups[train_index], groups[validation_index]).size:
        raise RuntimeError(
            "Session leakage detected between training and validation."
        )

    train_data = (
        X[train_index], y_class[train_index],
        reconstruction_reference[train_index],
        files[train_index], starts[train_index], groups[train_index],
    )

    validation_data = (
        X[validation_index], y_class[validation_index],
        reconstruction_reference[validation_index],
        files[validation_index], starts[validation_index],
        groups[validation_index],
    )

    return train_data, validation_data


In [ ]:
# ==========================================================
# CELL 10 — fit one fold/model (reconstruction-only)
# ==========================================================
def fit_model_with_group_validation(
    X_train_full, y_class_full, reconstruction_reference_full,
    files_full, starts_full, groups_full,
):
    train_data, validation_data = split_train_validation_by_group(
        X_train_full, y_class_full, reconstruction_reference_full,
        files_full, starts_full, groups_full,
    )

    (X_train, y_class_train, reconstruction_reference_train, _, _, _) = train_data
    (X_validation, y_class_validation, reconstruction_reference_validation,
     _, _, _) = validation_data

    normalization_mean, normalization_variance = (
        calculate_normalization_statistics(X_train)
    )

    X_train_normalized = normalize_windows_numpy(
        X_train, normalization_mean, normalization_variance,
    )
    X_validation_normalized = normalize_windows_numpy(
        X_validation, normalization_mean, normalization_variance,
    )

    model = build_autoencoder_model(normalization_mean, normalization_variance)

    sample_weights = make_training_sample_weights(reconstruction_reference_train)

    validation_sample_weights = (
        reconstruction_reference_validation.astype(np.float32)
    )

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", mode="min", patience=20,
            restore_best_weights=True, verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", mode="min", factor=0.5,
            patience=7, min_lr=1e-5, verbose=1,
        ),
    ]

    history = model.fit(
        X_train,
        X_train_normalized,
        sample_weight=sample_weights,
        validation_data=(
            X_validation,
            X_validation_normalized,
            validation_sample_weights,
        ),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=2,
        shuffle=True,
    )

    return model, normalization_mean, normalization_variance, history

In [ ]:

# ==========================================================
# CELL 11 — conditional reconstruction thresholds + anomaly
# decision (no class confidence, no anomaly-head probability —
# only reconstruction MSE vs a state/PWM/load-conditioned limit)
# ==========================================================
def get_pwm_bin_index(mean_pwm):
    bin_index = np.searchsorted(PWM_BIN_EDGES, mean_pwm, side="right") - 1
    return int(np.clip(bin_index, 0, len(PWM_BIN_NAMES) - 1))


def get_conditional_reconstruction_threshold(true_label_id, mean_pwm, mean_irms):
    """
    Select the healthy reconstruction limit for the current operating
    context. There is no classifier in this pipeline, so the KNOWN
    session label is used directly (this is an evaluation/calibration
    script, not a real-time fused classifier+AE system).

    motor_off / turning_off / turning_on:
        state-specific held-out safe threshold.

    normal or any fault label:
        compare against steady healthy operation at the same PWM bin
        and Irms-derived load state.
    """
    true_label = FINAL_LABELS[int(true_label_id)]

    if true_label in ["motor_off", "turning_off", "turning_on"]:
        return float(STATE_RECON_THRESHOLDS.get(true_label, RECON_THRESHOLD))

    bin_index = get_pwm_bin_index(mean_pwm)
    bin_name = PWM_BIN_NAMES[bin_index]

    load_split = float(
        PWM_LOAD_SPLITS.get(bin_name, GLOBAL_NORMAL_LOAD_SPLIT)
    )

    load_state = "high_load" if mean_irms >= load_split else "low_load"

    return float(
        NORMAL_PWM_LOAD_RECON_THRESHOLDS.get(bin_name, {}).get(
            load_state,
            NORMAL_PWM_RECON_THRESHOLDS.get(bin_name, RECON_THRESHOLD),
        )
    )


def flag_reconstruction_anomalies(reconstruction_mse, reconstruction_thresholds):
    return (reconstruction_mse > reconstruction_thresholds).astype(np.int64)


In [ ]:
# ==========================================================
# CELL 12 — five-fold file-wise cross-validation
# ==========================================================
cv_table = pd.DataFrame({
    "label": [session["class_id"] for session in sessions],
    "group": [session["session_id"] for session in sessions],
})

splitter = StratifiedGroupKFold(
    n_splits=N_SPLITS, shuffle=True, random_state=SEED,
)

cv_metrics = []
all_true_class = []
all_anomaly_true = []
all_reconstruction_mse = []
all_reconstruction_reference = []
all_mean_pwm = []
all_mean_irms = []
all_window_predictions = []
cv_best_epochs = []

for fold, (train_index, test_index) in enumerate(
    splitter.split(cv_table, cv_table["label"], groups=cv_table["group"]),
    start=1,
):
    print("\n" + "=" * 70)
    print("FOLD", fold)
    print("=" * 70)

    train_sessions = [sessions[index] for index in train_index]
    test_sessions = [sessions[index] for index in test_index]

    (
        X_train_full, y_class_train_full, _, reconstruction_reference_train_full,
        train_files, train_starts, train_groups,
    ) = build_dataset_from_sessions(train_sessions)

    (
        X_test, y_class_test, y_anomaly_test, reconstruction_reference_test,
        test_files, test_starts, _,
    ) = build_dataset_from_sessions(test_sessions)

    model, normalization_mean, normalization_variance, history = (
        fit_model_with_group_validation(
            X_train_full, y_class_train_full, reconstruction_reference_train_full,
            train_files, train_starts, train_groups,
        )
    )

    validation_loss_history = history.history.get("val_loss", [])
    if validation_loss_history:
        best_epoch = int(np.argmin(validation_loss_history) + 1)
    else:
        best_epoch = len(history.history.get("loss", []))

    cv_best_epochs.append(best_epoch)
    print("Best validation epoch:", best_epoch)

    X_test_normalized = normalize_windows_numpy(
        X_test, normalization_mean, normalization_variance,
    )

    reconstructed = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)

    reconstruction_mse = np.mean(
        np.square(reconstructed - X_test_normalized), axis=(1, 2),
    )

    pwm_feature_index = FEATURE_COLUMNS.index("PWM")
    irms_feature_index = FEATURE_COLUMNS.index(LOAD_PROXY_FEATURE)

    test_window_mean_pwm = np.mean(
        X_test[:, :, pwm_feature_index], axis=1,
    ).astype(np.float32)

    test_window_mean_irms = np.mean(
        X_test[:, :, irms_feature_index], axis=1,
    ).astype(np.float32)

    all_true_class.extend(y_class_test.tolist())
    all_anomaly_true.extend(y_anomaly_test.astype(np.int64).tolist())
    all_reconstruction_mse.extend(reconstruction_mse.tolist())
    all_reconstruction_reference.extend(
        reconstruction_reference_test.astype(np.int64).tolist()
    )
    all_mean_pwm.extend(test_window_mean_pwm.tolist())
    all_mean_irms.extend(test_window_mean_irms.tolist())

    fold_metrics = {
        "fold": fold,
        "mean_reconstruction_mse": float(np.mean(reconstruction_mse)),
    }
    cv_metrics.append(fold_metrics)

    fold_predictions = pd.DataFrame({
        "fold": fold,
        "file": test_files,
        "start_row": test_starts,
        "mean_pwm": test_window_mean_pwm,
        "mean_irms": test_window_mean_irms,
        "true_class_id": y_class_test,
        "true_label": [FINAL_LABELS[index] for index in y_class_test],
        "true_anomaly": y_anomaly_test.astype(np.int64),
        "reconstruction_reference": reconstruction_reference_test.astype(np.int64),
        "reconstruction_mse": reconstruction_mse,
    })
    all_window_predictions.append(fold_predictions)

    print(pd.Series(fold_metrics))

    history_df = pd.DataFrame(history.history)
    history_df.to_csv(
        os.path.join(CV_DIR, f"fold_{fold}_training_history.csv"), index=False,
    )

    tf.keras.backend.clear_session()

In [ ]:
# ==========================================================
# CELL 13 — combined CV report + conditional threshold
# calibration (state-specific + PWM/load-conditioned)
# ==========================================================
cv_metrics_df = pd.DataFrame(cv_metrics)
cv_predictions_df = pd.concat(all_window_predictions, ignore_index=True)

cv_metrics_df.to_csv(os.path.join(CV_DIR, "cv_metrics.csv"), index=False)
cv_predictions_df.to_csv(
    os.path.join(CV_DIR, "cv_window_predictions.csv"), index=False,
)

all_true_class = np.asarray(all_true_class, dtype=np.int64)
all_anomaly_true = np.asarray(all_anomaly_true, dtype=np.int64)
all_mean_pwm = np.asarray(all_mean_pwm, dtype=np.float32)
all_mean_irms = np.asarray(all_mean_irms, dtype=np.float32)
all_reconstruction_mse = np.asarray(all_reconstruction_mse, dtype=np.float32)
all_reconstruction_reference = np.asarray(
    all_reconstruction_reference, dtype=np.int64,
)

# HELD-OUT SAFE THRESHOLDS:
#   1) state-specific thresholds for motor_off / transitions
#   2) PWM + load-conditioned thresholds for steady normal
loaded_normal_mask = (all_reconstruction_reference == 1)

if not np.any(loaded_normal_mask):
    raise RuntimeError(
        "No held-out normal-with-load windows are available "
        "for reconstruction-threshold calibration."
    )

loaded_normal_reconstruction_mse = all_reconstruction_mse[loaded_normal_mask]

RECON_THRESHOLD = float(
    np.quantile(loaded_normal_reconstruction_mse, RECON_QUANTILE)
)

STATE_RECON_THRESHOLDS = {}

for state_label in ["motor_off", "turning_off", "turning_on"]:
    state_class_id = LABEL_TO_ID[state_label]
    state_errors = all_reconstruction_mse[all_true_class == state_class_id]

    if len(state_errors) >= MIN_THRESHOLD_WINDOWS:
        state_threshold = float(np.quantile(state_errors, RECON_QUANTILE))
    else:
        state_threshold = RECON_THRESHOLD
        print(
            f"WARNING: state {state_label} has only "
            f"{len(state_errors)} held-out windows. "
            "Using global safe fallback threshold."
        )

    STATE_RECON_THRESHOLDS[state_label] = state_threshold
    print(
        f"State {state_label} | windows={len(state_errors)} | "
        f"threshold={state_threshold:.8f}"
    )

normal_mask = loaded_normal_mask
normal_reconstruction_mse = all_reconstruction_mse[normal_mask]
normal_mean_pwm = all_mean_pwm[normal_mask]
normal_mean_irms = all_mean_irms[normal_mask]

GLOBAL_NORMAL_LOAD_SPLIT = float(
    np.quantile(normal_mean_irms, LOAD_SPLIT_QUANTILE)
)
GLOBAL_NORMAL_THRESHOLD = float(
    np.quantile(normal_reconstruction_mse, RECON_QUANTILE)
)

PWM_LOAD_SPLITS = {}
NORMAL_PWM_RECON_THRESHOLDS = {}
NORMAL_PWM_LOAD_RECON_THRESHOLDS = {}
threshold_rows = []

for bin_index, bin_name in enumerate(PWM_BIN_NAMES):
    lower_pwm = PWM_BIN_EDGES[bin_index]
    upper_pwm = PWM_BIN_EDGES[bin_index + 1]

    pwm_mask = (normal_mean_pwm >= lower_pwm) & (normal_mean_pwm < upper_pwm)
    pwm_errors = normal_reconstruction_mse[pwm_mask]
    pwm_irms = normal_mean_irms[pwm_mask]

    if len(pwm_errors) >= MIN_THRESHOLD_WINDOWS:
        pwm_threshold = float(np.quantile(pwm_errors, RECON_QUANTILE))
        load_split = float(np.quantile(pwm_irms, LOAD_SPLIT_QUANTILE))
    else:
        pwm_threshold = GLOBAL_NORMAL_THRESHOLD
        load_split = GLOBAL_NORMAL_LOAD_SPLIT
        print(
            f"WARNING: PWM bin {bin_name} has only "
            f"{len(pwm_errors)} normal windows. "
            "Using global normal fallback values."
        )

    PWM_LOAD_SPLITS[bin_name] = load_split
    NORMAL_PWM_RECON_THRESHOLDS[bin_name] = pwm_threshold
    NORMAL_PWM_LOAD_RECON_THRESHOLDS[bin_name] = {}

    for load_state in LOAD_STATE_NAMES:
        if load_state == "low_load":
            load_mask = pwm_mask & (normal_mean_irms < load_split)
        else:
            load_mask = pwm_mask & (normal_mean_irms >= load_split)

        load_errors = normal_reconstruction_mse[load_mask]

        if len(load_errors) >= MIN_THRESHOLD_WINDOWS:
            load_threshold = float(np.quantile(load_errors, RECON_QUANTILE))
        else:
            load_threshold = pwm_threshold
            print(
                f"WARNING: {bin_name}/{load_state} has only "
                f"{len(load_errors)} normal windows. "
                "Using its PWM-bin threshold."
            )

        NORMAL_PWM_LOAD_RECON_THRESHOLDS[bin_name][load_state] = load_threshold

        threshold_rows.append({
            "operating_state": "normal",
            "pwm_bin": bin_name,
            "lower_pwm": float(lower_pwm),
            "upper_pwm": float(upper_pwm),
            "load_state": load_state,
            "irms_split": float(load_split),
            "normal_windows": int(len(load_errors)),
            "reconstruction_threshold": float(load_threshold),
        })

        print(
            f"Normal {bin_name}/{load_state} | "
            f"windows={len(load_errors)} | "
            f"Irms split={load_split:.8f} | "
            f"threshold={load_threshold:.8f}"
        )

threshold_table = pd.DataFrame(threshold_rows)
threshold_table.to_csv(
    os.path.join(CV_DIR, "conditional_reconstruction_thresholds.csv"), index=False,
)

print("\nGlobal safe reconstruction fallback:", RECON_THRESHOLD)

# Apply the conditional thresholds window-by-window and score
# anomaly detection quality against the known fault/safe labels.
selected_thresholds = np.asarray([
    get_conditional_reconstruction_threshold(label_id, pwm, irms)
    for label_id, pwm, irms in zip(all_true_class, all_mean_pwm, all_mean_irms)
], dtype=np.float32)

all_anomaly_pred = flag_reconstruction_anomalies(
    all_reconstruction_mse, selected_thresholds,
)

overall_metrics = {
    "anomaly_accuracy": accuracy_score(all_anomaly_true, all_anomaly_pred),
    "anomaly_precision": precision_score(
        all_anomaly_true, all_anomaly_pred, zero_division=0,
    ),
    "anomaly_recall": recall_score(
        all_anomaly_true, all_anomaly_pred, zero_division=0,
    ),
    "anomaly_f1": f1_score(all_anomaly_true, all_anomaly_pred, zero_division=0),
    "anomaly_roc_auc": (
        roc_auc_score(all_anomaly_true, all_reconstruction_mse)
        if len(np.unique(all_anomaly_true)) == 2 else np.nan
    ),
    "reconstruction_threshold_global": RECON_THRESHOLD,
}

print("\nOVERALL CROSS-VALIDATION ANOMALY-DETECTION RESULTS")
print(pd.Series(overall_metrics))

print(
    classification_report(
        all_anomaly_true, all_anomaly_pred,
        target_names=["safe", "fault"], zero_division=0,
    )
)

pd.DataFrame([overall_metrics]).to_csv(
    os.path.join(CV_DIR, "overall_metrics.csv"), index=False,
)

ConfusionMatrixDisplay(
    confusion_matrix(all_anomaly_true, all_anomaly_pred, labels=[0, 1]),
    display_labels=["safe", "fault"],
).plot(values_format="d")

plt.title("Pure Autoencoder — Conditional Reconstruction Anomaly Detection")
plt.tight_layout()
plt.savefig(os.path.join(CV_DIR, "confusion_matrix.png"), dpi=180)
plt.show()


In [ ]:
# ==========================================================
# CELL 14 — train final model on all sessions
# ==========================================================
(
    X_all, y_class_all, y_anomaly_all, reconstruction_reference_all,
    all_files, all_starts, all_groups,
) = build_dataset_from_sessions(sessions)

final_mean, final_variance = calculate_normalization_statistics(X_all)
X_all_normalized = normalize_windows_numpy(X_all, final_mean, final_variance)

final_model = build_autoencoder_model(final_mean, final_variance)
final_sample_weights = make_training_sample_weights(reconstruction_reference_all)

FINAL_EPOCHS = max(int(np.median(cv_best_epochs)), 1)

print("\n" + "=" * 70)
print("FINAL DEPLOYMENT TRAINING ON 100% OF NORMAL-WITH-LOAD DATA")
print("=" * 70)
print("Sessions used:", len(np.unique(all_groups)))
print("Windows used:", len(X_all))
print("Final epochs from median CV best epoch:", FINAL_EPOCHS)

final_history = final_model.fit(
    X_all,
    X_all_normalized,
    sample_weight=final_sample_weights,
    epochs=FINAL_EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=2,
    shuffle=True,
)

pd.DataFrame(final_history.history).to_csv(
    os.path.join(FINAL_DIR, "final_training_history.csv"), index=False,
)


In [ ]:
# ==========================================================
# CELL 15 — save model, labels, metadata, thresholds
# ==========================================================
KERAS_PATH = os.path.abspath(
    os.path.join(FINAL_DIR, "pure_dense_autoencoder.keras")
)
H5_PATH = os.path.abspath(
    os.path.join(FINAL_DIR, "pure_dense_autoencoder.h5")
)
LABELS_PATH = os.path.abspath(os.path.join(FINAL_DIR, "labels.txt"))
METADATA_PATH = os.path.abspath(os.path.join(FINAL_DIR, "metadata.json"))
THRESHOLDS_PATH = os.path.abspath(os.path.join(FINAL_DIR, "thresholds.json"))

final_model.save(KERAS_PATH)
final_model.save(H5_PATH)

with open(LABELS_PATH, "w", encoding="utf-8") as file:
    file.write("\n".join(FINAL_LABELS))

thresholds = {
    "global_reconstruction_fallback": RECON_THRESHOLD,
    "state_reconstruction_thresholds": STATE_RECON_THRESHOLDS,
    "pwm_bin_edges": PWM_BIN_EDGES.tolist(),
    "pwm_bin_names": PWM_BIN_NAMES,
    "load_proxy_feature": LOAD_PROXY_FEATURE,
    "global_normal_load_split": GLOBAL_NORMAL_LOAD_SPLIT,
    "pwm_load_splits": PWM_LOAD_SPLITS,
    "normal_pwm_reconstruction_thresholds": NORMAL_PWM_RECON_THRESHOLDS,
    "normal_pwm_load_reconstruction_thresholds":
        NORMAL_PWM_LOAD_RECON_THRESHOLDS,
}

metadata = {
    "model_name": "pure_dense_autoencoder",
    "input_shape": [WINDOW_LEN, N_FEATURES],
    "feature_columns": FEATURE_COLUMNS,
    "labels": FINAL_LABELS,
    "safe_labels": SAFE_LABELS,
    "fault_labels": FAULT_LABELS,
    "scale_divisor": SCALE_DIVISOR,
    "window_step": WINDOW_STEP,
    "row_period_sec": ROW_PERIOD_SEC,
    "parameter_count": int(final_model.count_params()),
    "normalization_mean": final_mean.tolist(),
    "normalization_variance": final_variance.tolist(),
    "final_training_epochs": FINAL_EPOCHS,
    "thresholds": thresholds,
}

with open(METADATA_PATH, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

with open(THRESHOLDS_PATH, "w", encoding="utf-8") as file:
    json.dump(thresholds, file, indent=2)

print("Saved:", KERAS_PATH)
print("Saved:", H5_PATH)
print("Saved:", LABELS_PATH)
print("Saved:", METADATA_PATH)
print("Saved:", THRESHOLDS_PATH)


In [ ]:
# ==========================================================
# CELL 16 — export FLOAT32 and INT8 TFLite
# ==========================================================
TFLITE_FLOAT32_PATH = os.path.abspath(
    os.path.join(FINAL_DIR, "pure_dense_autoencoder_float32.tflite")
)

float32_converter = tf.lite.TFLiteConverter.from_keras_model(final_model)
float32_converter.optimizations = []
tflite_float32_model = float32_converter.convert()

with open(TFLITE_FLOAT32_PATH, "wb") as file:
    file.write(tflite_float32_model)

print("Saved FLOAT32 TFLite:", TFLITE_FLOAT32_PATH)
print(
    "FLOAT32 TFLite size KB:",
    round(os.path.getsize(TFLITE_FLOAT32_PATH) / 1024, 2),
)

TFLITE_INT8_PATH = os.path.abspath(
    os.path.join(FINAL_DIR, "pure_dense_autoencoder_int8.tflite")
)


def representative_dataset():
    sample_count = min(500, len(X_all))
    sample_indices = np.linspace(
        0, len(X_all) - 1, sample_count, dtype=np.int64,
    )
    for index in sample_indices:
        yield [X_all[index:index + 1].astype(np.float32)]


int8_converter = tf.lite.TFLiteConverter.from_keras_model(final_model)
int8_converter.optimizations = [tf.lite.Optimize.DEFAULT]
int8_converter.representative_dataset = representative_dataset
int8_converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8
]
int8_converter.inference_input_type = tf.int8
int8_converter.inference_output_type = tf.int8

tflite_int8_model = int8_converter.convert()

with open(TFLITE_INT8_PATH, "wb") as file:
    file.write(tflite_int8_model)

tflite_size_kb = os.path.getsize(TFLITE_INT8_PATH) / 1024
print("Saved INT8 TFLite:", TFLITE_INT8_PATH)
print("INT8 TFLite size KB:", round(tflite_size_kb, 2))


In [ ]:
# ==========================================================
# CELL 17 — export complete model as one C header
# (binary INT8 TFLite embedded as a byte array — same
#  approach used by the hybrid model's model-header cell)
# ==========================================================
MODEL_HEADER_PATH = os.path.abspath(
    os.path.join(FINAL_DIR, "pure_dense_autoencoder_model.h")
)


def write_binary_model_header(binary_model_path, header_path, array_name):
    with open(binary_model_path, "rb") as file:
        model_bytes = file.read()

    include_guard = array_name.upper() + "_H"

    with open(header_path, "w", encoding="utf-8") as file:
        file.write(f"#ifndef {include_guard}\n")
        file.write(f"#define {include_guard}\n\n")
        file.write("#include <stdint.h>\n")
        file.write("#include <stddef.h>\n\n")
        file.write("#if defined(__GNUC__)\n")
        file.write("#define MODEL_ALIGN __attribute__((aligned(16)))\n")
        file.write("#else\n")
        file.write("#define MODEL_ALIGN\n")
        file.write("#endif\n\n")
        file.write(
            f"static const MODEL_ALIGN uint8_t {array_name}"
            f"[{len(model_bytes)}] = {{\n"
        )

        for start in range(0, len(model_bytes), 12):
            chunk = model_bytes[start:start + 12]
            line = ", ".join(f"0x{byte:02X}" for byte in chunk)
            file.write(f"    {line},\n")

        file.write("};\n\n")
        file.write(
            f"static const size_t {array_name}_len = {len(model_bytes)}u;\n\n"
        )
        file.write(f"#endif /* {include_guard} */\n")

    return len(model_bytes)


header_model_size = write_binary_model_header(
    binary_model_path=TFLITE_INT8_PATH,
    header_path=MODEL_HEADER_PATH,
    array_name="pure_dense_autoencoder_model",
)

print("Saved complete model header:", MODEL_HEADER_PATH)
print("Binary model size KB:", round(header_model_size / 1024, 2))


In [ ]:
# ==========================================================
# CELL 18 — export firmware parameters header
# (normalization + PWM/load-conditioned reconstruction
#  thresholds — no class/anomaly-head parameters, since
#  those heads do not exist in this model)
# ==========================================================
PARAMS_HEADER_PATH = os.path.abspath(
    os.path.join(FINAL_DIR, "autoencoder_model_params.h")
)


def c_float(value):
    text = f"{float(value):.9g}"
    if "." not in text and "e" not in text and "E" not in text:
        text += ".0"
    return f"{text}f"


normal_low_load_thresholds = np.asarray([
    NORMAL_PWM_LOAD_RECON_THRESHOLDS[bin_name]["low_load"]
    for bin_name in PWM_BIN_NAMES
], dtype=np.float32)

normal_high_load_thresholds = np.asarray([
    NORMAL_PWM_LOAD_RECON_THRESHOLDS[bin_name]["high_load"]
    for bin_name in PWM_BIN_NAMES
], dtype=np.float32)

pwm_load_split_values = np.asarray([
    PWM_LOAD_SPLITS[bin_name] for bin_name in PWM_BIN_NAMES
], dtype=np.float32)

with open(PARAMS_HEADER_PATH, "w", encoding="utf-8") as file:
    file.write("#ifndef AUTOENCODER_MODEL_PARAMS_H\n")
    file.write("#define AUTOENCODER_MODEL_PARAMS_H\n\n")
    file.write(f"#define AE_MODEL_WINDOW_LEN {WINDOW_LEN}\n")
    file.write(f"#define AE_MODEL_N_FEATURES {N_FEATURES}\n")
    file.write(
        f"#define AE_MODEL_INPUT_SIZE {WINDOW_LEN * N_FEATURES}\n"
    )
    file.write(f"#define AE_MODEL_N_PWM_BINS {len(PWM_BIN_NAMES)}\n")
    file.write(
        "#define AE_MODEL_LOAD_PROXY_FEATURE_INDEX "
        f"{FEATURE_COLUMNS.index(LOAD_PROXY_FEATURE)}\n"
    )
    file.write(
        "#define AE_MODEL_RECON_THRESHOLD "
        f"{c_float(RECON_THRESHOLD)}\n\n"
    )

    file.write(
        "static const float "
        f"AE_MODEL_NORMALIZATION_MEAN[{N_FEATURES}] = {{\n    "
    )
    file.write(", ".join(c_float(value) for value in final_mean))
    file.write("\n};\n\n")

    file.write(
        "static const float "
        f"AE_MODEL_NORMALIZATION_VARIANCE[{N_FEATURES}] = {{\n    "
    )
    file.write(", ".join(c_float(value) for value in final_variance))
    file.write("\n};\n\n")

    file.write(
        "static const float "
        f"AE_MODEL_PWM_BIN_EDGES[{len(PWM_BIN_EDGES)}] = {{\n    "
    )
    file.write(", ".join(c_float(value) for value in PWM_BIN_EDGES))
    file.write("\n};\n\n")

    file.write(
        "static const float "
        f"AE_MODEL_PWM_LOAD_SPLITS[{len(PWM_BIN_NAMES)}] = {{\n    "
    )
    file.write(", ".join(c_float(value) for value in pwm_load_split_values))
    file.write("\n};\n\n")

    file.write(
        "static const float "
        f"AE_MODEL_NORMAL_LOW_LOAD_THRESHOLDS[{len(PWM_BIN_NAMES)}] = {{\n    "
    )
    file.write(
        ", ".join(c_float(value) for value in normal_low_load_thresholds)
    )
    file.write("\n};\n\n")

    file.write(
        "static const float "
        f"AE_MODEL_NORMAL_HIGH_LOAD_THRESHOLDS[{len(PWM_BIN_NAMES)}] = {{\n    "
    )
    file.write(
        ", ".join(c_float(value) for value in normal_high_load_thresholds)
    )
    file.write("\n};\n\n")

    file.write(
        "#define AE_MODEL_MOTOR_OFF_RECON_THRESHOLD "
        f"{c_float(STATE_RECON_THRESHOLDS['motor_off'])}\n"
    )
    file.write(
        "#define AE_MODEL_TURNING_OFF_RECON_THRESHOLD "
        f"{c_float(STATE_RECON_THRESHOLDS['turning_off'])}\n"
    )
    file.write(
        "#define AE_MODEL_TURNING_ON_RECON_THRESHOLD "
        f"{c_float(STATE_RECON_THRESHOLDS['turning_on'])}\n\n"
    )

    file.write("#endif /* AUTOENCODER_MODEL_PARAMS_H */\n")

print("Saved firmware parameters header:", PARAMS_HEADER_PATH)


In [ ]:
# ==========================================================
# CELL 19 — verify TFLite model interfaces
# ==========================================================
interpreter = tf.lite.Interpreter(model_path=TFLITE_INT8_PATH)
interpreter.allocate_tensors()

print("TFLite input details:")
for detail in interpreter.get_input_details():
    print(detail)

print("TFLite output details:")
for detail in interpreter.get_output_details():
    print(detail)


In [ ]:
# ==========================================================
# CELL 19A — load saved model without retraining
# ==========================================================
KERAS_PATH = os.path.abspath(
    os.path.join(FINAL_DIR, "pure_dense_autoencoder.keras")
)

METADATA_PATH = os.path.abspath(
    os.path.join(FINAL_DIR, "metadata.json")
)

THRESHOLDS_PATH = os.path.abspath(
    os.path.join(FINAL_DIR, "thresholds.json")
)

required_files = [
    KERAS_PATH,
    METADATA_PATH,
    THRESHOLDS_PATH,
]

missing_files = [
    path for path in required_files
    if not os.path.exists(path)
]

if missing_files:
    raise FileNotFoundError(
        "Saved model files are missing. Train and save the model first:\n"
        + "\n".join(missing_files)
    )

# Load trained Keras model.
final_model = tf.keras.models.load_model(
    KERAS_PATH,
    compile=False,
)

# Load normalization parameters and model configuration.
with open(METADATA_PATH, "r", encoding="utf-8") as file:
    metadata = json.load(file)

final_mean = np.asarray(
    metadata["normalization_mean"],
    dtype=np.float32,
)

final_variance = np.asarray(
    metadata["normalization_variance"],
    dtype=np.float32,
)

# Load all reconstruction thresholds.
with open(THRESHOLDS_PATH, "r", encoding="utf-8") as file:
    thresholds = json.load(file)

RECON_THRESHOLD = float(
    thresholds["global_reconstruction_fallback"]
)

STATE_RECON_THRESHOLDS = {
    key: float(value)
    for key, value
    in thresholds["state_reconstruction_thresholds"].items()
}

PWM_BIN_EDGES = np.asarray(
    thresholds["pwm_bin_edges"],
    dtype=np.float32,
)

PWM_BIN_NAMES = list(
    thresholds["pwm_bin_names"]
)

LOAD_PROXY_FEATURE = thresholds["load_proxy_feature"]

GLOBAL_NORMAL_LOAD_SPLIT = float(
    thresholds["global_normal_load_split"]
)

PWM_LOAD_SPLITS = {
    key: float(value)
    for key, value
    in thresholds["pwm_load_splits"].items()
}

NORMAL_PWM_RECON_THRESHOLDS = {
    key: float(value)
    for key, value
    in thresholds[
        "normal_pwm_reconstruction_thresholds"
    ].items()
}

NORMAL_PWM_LOAD_RECON_THRESHOLDS = {
    pwm_bin: {
        load_state: float(value)
        for load_state, value in load_thresholds.items()
    }
    for pwm_bin, load_thresholds
    in thresholds[
        "normal_pwm_load_reconstruction_thresholds"
    ].items()
}

print("Saved autoencoder loaded successfully.")
print("Model:", KERAS_PATH)
print("Input shape:", final_model.input_shape)
print("Output shape:", final_model.output_shape)
print("Global reconstruction threshold:", RECON_THRESHOLD)

In [ ]:
# ==========================================================
# CELL 20 — manual reconstruction test
# ==========================================================
def manual_autoencoder_prediction(test_file):
    # model = tf.keras.models.load_model(KERAS_PATH, compile=False)

    test_df = read_one(test_file)

    X_test, starts = make_windows_from_array(
        test_df.to_numpy(dtype=np.float32)
    )

    if X_test is None or len(X_test) == 0:
        raise RuntimeError(
            "The selected file does not contain enough valid rows "
            "for one 30-row window."
        )

    X_test_normalized = normalize_windows_numpy(X_test, final_mean, final_variance)

    reconstructed = final_model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)

    reconstruction_mse = np.mean(
        np.square(reconstructed - X_test_normalized), axis=(1, 2),
    )

    pwm_feature_index = FEATURE_COLUMNS.index("PWM")
    irms_feature_index = FEATURE_COLUMNS.index(LOAD_PROXY_FEATURE)

    window_mean_pwm = np.mean(
        X_test[:, :, pwm_feature_index], axis=1,
    ).astype(np.float32)
    window_mean_irms = np.mean(
        X_test[:, :, irms_feature_index], axis=1,
    ).astype(np.float32)

    # No classifier exists, so use the file's own known label
    # (from label_for) to choose the correct threshold family,
    # exactly like the CV loop above.
    file_label = label_for(test_file)
    file_class_id = (
        LABEL_TO_ID[file_label] if file_label in LABEL_TO_ID
        else LABEL_TO_ID["normal"]
    )

    window_reconstruction_thresholds = np.asarray([
        get_conditional_reconstruction_threshold(file_class_id, mean_pwm, mean_irms)
        for mean_pwm, mean_irms in zip(window_mean_pwm, window_mean_irms)
    ], dtype=np.float32)

    anomaly_flags = flag_reconstruction_anomalies(
        reconstruction_mse, window_reconstruction_thresholds,
    )

    result = pd.DataFrame({
        "window": np.arange(len(X_test)),
        "start_row": starts,
        "time_sec": (starts + WINDOW_LEN / 2.0) * ROW_PERIOD_SEC,
        "mean_pwm": window_mean_pwm,
        "mean_irms": window_mean_irms,
        "reconstruction_mse": reconstruction_mse,
        "reconstruction_threshold": window_reconstruction_thresholds,
        "anomaly_flag": anomaly_flags,
    })

    stem = Path(test_file).stem
    output_csv = os.path.join(MANUAL_TEST_DIR, f"manual_test_{stem}.csv")
    result.to_csv(output_csv, index=False)

    print("\n" + "=" * 70)
    print("MANUAL PURE-AUTOENCODER RECONSTRUCTION TEST")
    print("=" * 70)
    print("File:", test_file)
    print("Assumed file label (for threshold selection):", file_label)
    print("Total windows:", len(result))
    print("Mean reconstruction MSE:", float(np.mean(reconstruction_mse)))
    print(
        "Anomalous windows:", int(np.sum(anomaly_flags)), "/", len(result),
    )
    print("Saved CSV:", output_csv)

    # ------------------------------------------------------
    # CHART — reconstruction MSE vs conditional threshold
    # ------------------------------------------------------
    plt.figure(figsize=(14, 5))

    plt.plot(
        result["time_sec"], reconstruction_mse,
        label="Reconstruction MSE", linewidth=2,
    )

    plt.plot(
        result["time_sec"], window_reconstruction_thresholds,
        linestyle="--", linewidth=2,
        label="State/PWM/load-conditioned threshold",
    )

    plt.fill_between(
        result["time_sec"], 0, reconstruction_mse,
        where=(reconstruction_mse > window_reconstruction_thresholds),
        alpha=0.15, label="Reconstruction anomaly",
    )

    plt.xlabel("Time [s]")
    plt.ylabel("Reconstruction MSE")
    plt.title(f"Pure Autoencoder reconstruction head — {stem}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    reconstruction_head_path = os.path.join(
        MANUAL_TEST_DIR, f"manual_test_{stem}_reconstruction_head.png",
    )
    plt.savefig(reconstruction_head_path, dpi=180)
    plt.show()
    plt.close()

    print("Saved reconstruction-head chart:", reconstruction_head_path)

    # ------------------------------------------------------
    # CHART — anomaly-flag timeline
    # ------------------------------------------------------
    plt.figure(figsize=(14, 4))

    plt.step(result["time_sec"], result["anomaly_flag"], where="mid", linewidth=2)
    plt.yticks([0, 1], ["safe", "anomaly"])
    plt.xlabel("Time [s]")
    plt.ylabel("Decision")
    plt.title(f"Anomaly decision timeline — {stem}")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    timeline_path = os.path.join(
        MANUAL_TEST_DIR, f"manual_test_{stem}_anomaly_timeline.png",
    )
    plt.savefig(timeline_path, dpi=180)
    plt.show()
    plt.close()

    print("Saved anomaly timeline:", timeline_path)

    return result

In [ ]:
# ==========================================================
# CELL 21 — select manual test file
# ==========================================================
import tkinter as tk
from tkinter import filedialog

root = tk.Tk()
root.withdraw()
root.attributes("-topmost", True)

TEST_FILE = filedialog.askopenfilename(
    title="Select motor CSV or Excel file",
    filetypes=[
        ("Excel files", "*.xlsx"),
        ("CSV files", "*.csv"),
        ("All files", "*.*"),
    ],
)

root.destroy()

if TEST_FILE:
    manual_result = manual_autoencoder_prediction(TEST_FILE)
else:
    print("No file selected.")
